# Committee Statistical Analysis -- (Synthetic Data)

This notebook demonstrates the statistical methods used to evaluate a
game-theoretic multi-agent LLM committee against ground truth, using
**entirely synthetic data** shaped like the real dataset.

The original analysis was run on SRTR patient data, which is restricted
under a data use agreement and cannot be shared. This notebook exists to
make the *methodology* reproducible and inspectable, not to reproduce the
original results.

In [1]:
from generate_synthetic_data import generate_synthetic_committee_data
from stats_analysis import (
    build_ppv_npv_table, pairwise_mcnemar, all_pairwise_bootstrap,
    run_subgroup_fairness_screen, add_fdr_correction
)

df = generate_synthetic_committee_data(n_patients=300, seed=0)
GAME_COLS = {"Normal": None, "Normal Co-Op": None}
DEMO_COLS = ["gender"]

df.head()

,label,vote_Normal,vote_Normal Co-Op,gender,ethnicity,education,diagnosis,age,adi_score,age_group,adi_group
0,1,0,1,Female,Group B,Associate/Bachelor,Diagnosis A,31,32.692824,18-34,Q2
1,1,1,1,Male,Group A,High School,Diagnosis B,20,59.370476,18-34,Q3
2,1,0,1,Male,Group B,High School,Diagnosis C,48,20.336734,35-49,Q1 (least deprived)
3,0,0,0,Female,Group A,Associate/Bachelor,Diagnosis A,58,45.922898,50-64,Q3
4,0,0,0,Male,Group B,High School,Diagnosis B,63,24.530362,50-64,Q2


In [2]:
import sklearn.metrics
import numpy as np
import pandas as pd
import statsmodels.stats.proportion as prop
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

### Confusion matrix values for each game
tn_normal, fp_normal, fn_normal, tp_normal = sklearn.metrics.confusion_matrix(
    df["label"], df["vote_Normal"]
).ravel().tolist()

tn_normal_coop, fp_normal_coop, fn_normal_coop, tp_normal_coop = sklearn.metrics.confusion_matrix(
    df["label"], df["vote_Normal Co-Op"]
).ravel().tolist()

## 1. PPV and NPV, with Wilson 95% CIs


In [12]:
#PPV and NPV tests with Wilsons 95% CI
import statsmodels.stats.proportion as prop
import numpy as np
### Normal
ppv_normal = tp_normal / (tp_normal + fp_normal)
ci_ppv_norm = prop.proportion_confint(tp_normal, (tp_normal + fp_normal), alpha=0.05, method='wilson')
npv_normal = tn_normal / (tn_normal + fn_normal)
ci_npv_norm = prop.proportion_confint(tn_normal, (tn_normal + fn_normal), alpha=0.05, method='wilson')
### Normal Co-Op
ppv_normal_coop = tp_normal_coop / (tp_normal_coop + fp_normal_coop)
ci_ppv_norm_c = prop.proportion_confint(tp_normal_coop, (tp_normal_coop + fp_normal_coop), alpha=0.05, method='wilson')
npv_normal_coop = tn_normal_coop / (tn_normal_coop + fn_normal_coop)
ci_npv_norm_c = prop.proportion_confint(tn_normal_coop, (tn_normal_coop + fn_normal_coop), alpha=0.05, method='wilson')

f_norm_ppv = f"{ppv_normal} ({ci_ppv_norm[0]:.2f},{ci_ppv_norm[1]:.2f})"
f_norm_npv = f"{npv_normal} ({ci_npv_norm[0]:.2f},{ci_npv_norm[1]:.2f})"
f_norm_c_ppv = f"{ppv_normal_coop} ({ci_ppv_norm_c[0]:.2f},{ci_ppv_norm_c[1]:.2f})"
f_norm_c_npv = f"{npv_normal_coop} ({ci_npv_norm_c[0]:.2f},{ci_npv_norm_c[1]:.2f})"

print("Normal   -> PPV:", f_norm_ppv, " NPV:", f_norm_npv)
print("Normal Co-Op -> PPV:", f_norm_c_ppv, " NPV:", f_norm_c_npv)

Normal   -> PPV: 0.9230769230769231 (0.87,0.96)  NPV: 0.8680555555555556 (0.80,0.91)
Normal Co-Op -> PPV: 0.9433962264150944 (0.90,0.97)  NPV: 0.9078014184397163 (0.85,0.95)


## 2. McNemar's Test

Since both games score the *same* patients, this is a paired comparison:
it isolates the patients where the two games disagreed, and tests whether
one game won those disagreements meaningfully more often than the other.

In [3]:
# McNemar Test: for any differences between the Normal Vs Normal Coop Games. 
# it looks specifically at the patients where the two games disagree, and
# asks whether one game wins those disagreements more often than the other.

correct_normal = (df["vote_Normal"] == df["label"]).astype(int)
correct_normal_coop = (df["vote_Normal Co-Op"] == df["label"]).astype(int)

# Build the 2x2 table: rows = Normal correct/incorrect, cols = Co-Op correct/incorrect
mcnemar_table = pd.crosstab(correct_normal, correct_normal_coop).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0
).values

both_wrong      = mcnemar_table[0, 0]
normal_only     = mcnemar_table[1, 0]   # Normal right, Co-Op wrong
coop_only       = mcnemar_table[0, 1]   # Co-Op right, Normal wrong
both_right      = mcnemar_table[1, 1]

print("Both wrong:", both_wrong)
print("Normal right, Co-Op wrong:", normal_only)
print("Co-Op right, Normal wrong:", coop_only)
print("Both right:", both_right)

# exact=True uses the exact binomial test, the safer choice when the
# discordant counts (normal_only, coop_only) might be small
mcnemar_result = mcnemar(mcnemar_table, exact=True)

print(f"\nMcNemar's test statistic: {mcnemar_result.statistic}")
print(f"p-value: {mcnemar_result.pvalue}")

if mcnemar_result.pvalue < 0.05:
    print("\n-> Significant difference between Normal and Normal Co-Op (p < 0.05)")
else:
    print("\n-> No significant difference detected between Normal and Normal Co-Op")

#Paired bootstrap CI: how big is the gap between Normal and Normal Co-Op? this tells you the actual size of the difference from the McNemar test, 
#which might have a significant difference when running the full dataset

def accuracy(label, vote):
    return (label == vote).mean()

rng = np.random.default_rng(42)   # fixed seed so results are reproducible
n_boot = 5000
n = len(df)

label_arr = df["label"].values
vote_normal_arr = df["vote_Normal"].values
vote_coop_arr = df["vote_Normal Co-Op"].values

observed_diff = accuracy(label_arr, vote_coop_arr) - accuracy(label_arr, vote_normal_arr)

boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.integers(0, n, n)   # resample patient indices with replacement
    boot_diffs[i] = (accuracy(label_arr[idx], vote_coop_arr[idx])
                      - accuracy(label_arr[idx], vote_normal_arr[idx]))

ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

print(f"\nObserved accuracy difference (Co-Op - Normal): {observed_diff:.4f}")
print(f"95% bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")

if ci_low > 0 or ci_high < 0:
    print("\n-> CI excludes zero: evidence of a real difference")
else:
    print("\n-> CI includes zero: no evidence of a real difference")

Both wrong: 2
Normal right, Co-Op wrong: 20
Co-Op right, Normal wrong: 29
Both right: 249

McNemar's test statistic: 20.0
p-value: 0.2528697301676033

-> No significant difference detected between Normal and Normal Co-Op

Observed accuracy difference (Co-Op - Normal): 0.0300
95% bootstrap CI: [-0.0167, 0.0733]

-> CI includes zero: no evidence of a real difference


## 4. Subgroup Fairness Analysis

Checks whether any demographic subgroup is being treated systematically
worse by the committee's decisions. For each subgroup, false positive and
false negative rates are compared against the rest of the cohort using a
two-proportion z-test, run separately per game. Splitting by error type
matters clinically: a disparity in false negatives (wrongly denying a
transplant) carries very different consequences than a disparity in false
positives (wrongly approving one), so collapsing them into a single "error
rate" would obscure which kind of harm a subgroup is disproportionately
exposed to.


**Note:** in the original analysis, this section built `df` directly from
the real SRTR dataset and the committee's `all_comp` output array:

```python
patient_no = all_comp.shape[0]

df = SRTR_Data.iloc[:patient_no][DEMO_COLS].copy()
df["label"] = all_comp[:patient_no, 0].astype(int)

df["age_group"] = pd.cut(
    SRTR_Data.iloc[:patient_no]["CAN_AGE_AT_LISTING"],
    bins=[0, 18, 35, 50, 65, 100],
    labels=["<18", "18-34", "35-49", "50-64", "65+"]
)
df["adi_group"] = pd.qcut(
    SRTR_Data.iloc[:patient_no]["zip_ADI"],
    q=4,
    labels=["Q1 (least deprived)", "Q2", "Q3", "Q4 (most deprived)"]
)

for game_name, col in GAME_COLS.items():
    df[f"vote_{game_name}"] = all_comp[:patient_no, col].astype(int)
```

Since SRTR data cannot be shared, the cell below reconstructs an
equivalent `df` using the synthetic dataset instead.


In [4]:
###subgroup analysis
import numpy as np
import pandas as pd
import sklearn.metrics
import statsmodels.stats.proportion as prop
from statsmodels.stats.proportion import proportions_ztest




GAME_COLS = {
    "Normal": None,
    "Normal Co-Op": None,
}

DEMO_COLS = [
    "gender",
    "ethnicity",
    "education",
    "diagnosis",
]

#build a table of each patients ground truth value and what each game voted.



def error_type(label, vote):
    """Return TP / TN / FP / FN for a single (ground-truth, predicted) pair."""
    if label == 1 and vote == 1:
        return "TP"
    if label == 0 and vote == 0:
        return "TN"
    if label == 0 and vote == 1:
        return "FP"
    return "FN"


for game_name in GAME_COLS:
    df[f"err_{game_name}"] = [
        error_type(l, v) for l, v in zip(df["label"], df[f"vote_{game_name}"])
    ]

# ---------------------------------------------------------------------------
# 2. Per-subgroup metrics for each game, with Wilson CIs
#    (reuses the same sensitivity/specificity/accuracy logic you already had,
#    just applied to each subgroup slice instead of the whole cohort)
# ---------------------------------------------------------------------------

def subgroup_metrics(sub_df, game_name):
    label = sub_df["label"].values
    vote = sub_df[f"vote_{game_name}"].values
    if len(np.unique(label)) < 2 or len(sub_df) == 0:
        return None  # not enough data to form a 2x2 table

    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(
        label, vote, labels=[0, 1]
    ).ravel().tolist()

    n_pos = tp + fn
    n_neg = tn + fp
    n_tot = n_pos + n_neg

    out = {"n": n_tot}

    if n_pos > 0:
        sens = tp / n_pos
        ci = prop.proportion_confint(tp, n_pos, alpha=0.05, method="wilson")
        out["sensitivity"] = sens
        out["sens_ci"] = ci
        out["fn_rate"] = fn / n_pos
    if n_neg > 0:
        spec = tn / n_neg
        ci = prop.proportion_confint(tn, n_neg, alpha=0.05, method="wilson")
        out["specificity"] = spec
        out["spec_ci"] = ci
        out["fp_rate"] = fp / n_neg
    if n_tot > 0:
        acc = (tp + tn) / n_tot
        ci = prop.proportion_confint(tp + tn, n_tot, alpha=0.05, method="wilson")
        out["accuracy"] = acc
        out["acc_ci"] = ci

    return out


def disparity_table(df, demo_col):
    """Build a table of per-subgroup, per-game metrics for one demographic
    variable, so you can eyeball whether a subgroup's error rate is worse
    and whether that gap shrinks or persists across game types."""
    rows = []
    for level, sub_df in df.groupby(demo_col):
        row = {"variable": demo_col, "level": level}
        for game_name in GAME_COLS:
            m = subgroup_metrics(sub_df, game_name)
            if m is None:
                continue
            row[f"{game_name}_n"] = m["n"]
            row[f"{game_name}_fp_rate"] = m.get("fp_rate")
            row[f"{game_name}_fn_rate"] = m.get("fn_rate")
            row[f"{game_name}_accuracy"] = m.get("accuracy")
        rows.append(row)
    return pd.DataFrame(rows)


all_disparity_tables = {}
for demo_col in DEMO_COLS:
    all_disparity_tables[demo_col] = disparity_table(df, demo_col)
    print(f"\n=== Disparity table: {demo_col} ===")
    print(all_disparity_tables[demo_col].to_string(index=False))

# ---------------------------------------------------------------------------
# 3. Formal test: is a given subgroup's error rate significantly different
#    from the rest of the cohort, per game? (two-proportion z-test)
#    This is the piece that lets you say "the disparity for X is significant
#    under the Normal game but not under Co-Op", etc.
# ---------------------------------------------------------------------------

def subgroup_vs_rest_test(df, demo_col, level, game_name, error_col="FP"):
    """error_col: 'FP' or 'FN' -- which error type's rate to compare."""
    in_group = df[demo_col] == level
    err_flags = (df[f"err_{game_name}"] == error_col).astype(int)

    # denominator = patients eligible for this error type
    # FP denominator = true negatives (label==0), FN denominator = true positives (label==1)
    eligible_label = 0 if error_col == "FP" else 1
    eligible = df["label"] == eligible_label

    g_count = int((err_flags[in_group & eligible]).sum())
    g_n = int((in_group & eligible).sum())
    rest_count = int((err_flags[~in_group & eligible]).sum())
    rest_n = int((~in_group & eligible).sum())

    if g_n == 0 or rest_n == 0:
        return None

    stat, pval = proportions_ztest(
        count=[g_count, rest_count], nobs=[g_n, rest_n]
    )
    return {
        "demo_col": demo_col,
        "level": level,
        "game": game_name,
        "error_type": error_col,
        "group_rate": g_count / g_n,
        "rest_rate": rest_count / rest_n,
        "p_value": pval,
    }


# Example: run this for every level of every demographic variable, every
# game, both FP and FN -- flags anything p < 0.05 for a closer look.
results = []
for demo_col in DEMO_COLS:
    for level in df[demo_col].dropna().unique():
        for game_name in GAME_COLS:
            for error_col in ["FP", "FN"]:
                r = subgroup_vs_rest_test(df, demo_col, level, game_name, error_col)
                if r is not None:
                    results.append(r)

results_df = pd.DataFrame(results).sort_values("p_value")
results_df = results_df[~results_df["demo_col"].isin(["CAN_AGE_AT_LISTING", "zip_ADI"])]
print("\n=== Subgroup vs. rest of cohort (sorted by p-value) ===")
print(results_df.to_string(index=False))

flagged = results_df[results_df["p_value"] < 0.05]
print("\n=== Statistically significant disparities (p < 0.05) ===")
print(flagged.to_string(index=False))




=== Disparity table: gender ===
variable  level  Normal_n  Normal_fp_rate  Normal_fn_rate  Normal_accuracy  Normal Co-Op_n  Normal Co-Op_fp_rate  Normal Co-Op_fn_rate  Normal Co-Op_accuracy
  gender Female       150        0.121622        0.131579         0.873333             150              0.081081              0.065789               0.926667
  gender   Male       150        0.047619        0.103448         0.920000             150              0.047619              0.091954               0.926667

=== Disparity table: ethnicity ===
 variable   level  Normal_n  Normal_fp_rate  Normal_fn_rate  Normal_accuracy  Normal Co-Op_n  Normal Co-Op_fp_rate  Normal Co-Op_fn_rate  Normal Co-Op_accuracy
ethnicity Group A       210        0.096154        0.132075         0.885714             210              0.067308              0.075472               0.928571
ethnicity Group B        90        0.060606        0.087719         0.922222              90              0.060606              0.087719 

## 5. Benjamini-Hochberg (FDR) Correction

The subgroup fairness analysis above ran a separate statistical test for
every combination of demographic subgroup, game, and error type -- dozens
of tests in total. 

Running that many tests without correction means some tests are expected to cross p < 0.05 
purely by chance.The Benjamini-Hochberg procedure controls the expected proportion of false
positives among the flagged results, which is a more defensible standard
for exploratory fairness screening than treating each raw p-value alone.

In [5]:
### FDR (Benjamini-Hochberg) correction for the subgroup fairness tests

from statsmodels.stats.multitest import multipletests

reject, p_adj, _, _ = multipletests(results_df["p_value"], method="fdr_bh")

results_df["p_value_fdr"] = p_adj
results_df["significant_after_fdr"] = reject

print("\n=== All subgroup tests, sorted by FDR-corrected p-value ===")
print(results_df.sort_values("p_value_fdr").to_string(index=False))

flagged_fdr = results_df[results_df["significant_after_fdr"]]
print(f"\n=== Disparities that survive FDR correction (n = {len(flagged_fdr)}) ===")
print(flagged_fdr.to_string(index=False))

print(f"\nRaw p < 0.05 hits: {len(flagged)}")
print(f"Hits surviving FDR correction: {len(flagged_fdr)}")



=== All subgroup tests, sorted by FDR-corrected p-value ===
 demo_col              level         game error_type  group_rate  rest_rate  p_value  p_value_fdr  significant_after_fdr
education Associate/Bachelor Normal Co-Op         FP    0.130435   0.032967 0.029654     0.768836                  False
   gender             Female       Normal         FN    0.131579   0.103448 0.576627     0.768836                  False
   gender               Male       Normal         FN    0.103448   0.131579 0.576627     0.768836                  False
education Associate/Bachelor       Normal         FN    0.135593   0.105769 0.568527     0.768836                  False
   gender               Male Normal Co-Op         FN    0.091954   0.065789 0.538479     0.768836                  False
   gender             Female Normal Co-Op         FN    0.065789   0.091954 0.538479     0.768836                  False
education        High School       Normal         FP    0.108696   0.076923 0.534435     0.7